In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import spearmanr
import re

# LOAD DATA
DATA_DIR = Path("./patienten-visiten")
csv_files = sorted(DATA_DIR.glob("Patients_V*.csv"))

dfs = [pd.read_csv(f, sep=";") for f in csv_files]
data = pd.concat(dfs, ignore_index=True)
print("Rows:", len(data))

def get_visit(name):
    match = re.search(r"_K(\d+)$", str(name))
    return match.group(1) if match else None

def clean_name(name):
    name = str(name)
    name = re.sub(r"_K\d+$", "", name)
    name = name.replace("_K", "")
    return name

def to_numeric(s):
    return pd.to_numeric(s.astype(str).str.replace(",", "."), errors="coerce")

# SCORE + FACTOR BASE CLEANING
score_base = ["midas_result", "dass_depression", "dass_fear", "dass_stress", "gvas_result", "pgic_result", "chiq_result"]
factor_base = ["gender", "birthyear", "weight", "pregnant", "highest_degree", "family_status", "current_employed", "working_hours_model", "shiftwork", "working_hours_reduced", "children", "headache_days", "days_medication", "intensity", "days_lost", "days_doctor", "days_ER", "days_hospital"]

score_cols = [c for c in data.columns if any(c.startswith(x) for x in score_base)]
factor_cols = [c for c in data.columns if any(c.startswith(x) for x in factor_base)]

results = []

for score_col in score_cols:
    score_name = clean_name(score_col)
    score_visit = get_visit(score_col)
    score_data = to_numeric(data[score_col])
    
    for factor_col in factor_cols:
        factor_name = clean_name(factor_col)
        factor_visit = get_visit(factor_col)
        
        # Match visits: either they match or the factor is global (visit is None)
        if factor_visit is not None and score_visit != factor_visit:
            continue

        tmp = pd.DataFrame({
            "score": score_data,
            "factor": data[factor_col]
        }).dropna()
        
        if len(tmp) < 30:
            continue

        # Type detection
        ftmp = to_numeric(tmp["factor"])
        if ftmp.notna().mean() > 0.9:
            if ftmp.nunique() <= 2:
                val = ftmp.corr(tmp["score"])
                results.append({"score": score_name, "factor": factor_name, "type": "binary", "value": None, "association": val, "n": len(tmp)})
            else:
                val, _ = spearmanr(ftmp, tmp["score"], nan_policy='omit')
                results.append({"score": score_name, "factor": factor_name, "type": "numeric", "value": None, "association": val, "n": len(tmp)})
        else:
            global_mean = tmp["score"].mean()
            for cat_val, group in tmp.groupby("factor"):
                if len(group) >= 10:
                    results.append({"score": score_name, "factor": factor_name, "type": "categorical", "value": cat_val, "association": group["score"].mean() - global_mean, "n": len(group)})

df_results = pd.DataFrame(results)
df_results.to_csv("questionnaire_data_other_factors.csv", index=False)
print("Saved: questionnaire_data_other_factors.csv")
print(df_results.head())

C:\Users\karlw\AppData\Local\Temp\ipykernel_38760\1511868829.py:11: DtypeWarning: Columns (0: ichd3_4_K1, 1: comment_4_K1, 2: adverse2_a5_K1, 3: adverse3_a5_K1, 4: adverse1_a6_K1, 5: adverse2_a6_K1, 6: adverse3_a6_K1, 7: acute_name_a7_K1, 8: dosage_unit_a7_K1, 9: dosage_admin_a7_K1, 10: adverse1_a7_K1, 11: adverse2_a7_K1, 12: adverse3_a7_K1, 13: acute_name_a8_K1, 14: dosage_unit_a8_K1, 15: dosage_admin_a8_K1, 16: adverse1_a8_K1, 17: adverse2_a8_K1, 18: adverse3_a8_K1, 19: acute_name_a9_K1, 20: dosage_unit_a9_K1, 21: dosage_admin_a9_K1, 22: adverse1_a9_K1, 23: acute_name_a10_K1, 24: dosage_unit_a10_K1, 25: dosage_admin_a10_K1, 26: adverse1_a10_K1, 27: adverse2_a10_K1, 28: aborted_good_but_adverse_effect_fa_2_K1, 29: aborted_therapy_break_fa_2_K1, 30: aborted_not_needed_fa_2_K1, 31: aborted_good_but_adverse_effect_fa_3_K1, 32: aborted_therapy_break_fa_3_K1, 33: aborted_not_needed_fa_3_K1, 34: aborted_good_but_adverse_effect_fa_4_K1, 35: aborted_better_fa_4_K1, 36: aborted_therapy_break_f

Rows: 24497


C:\Users\karlw\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\karlw\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
C:\Users\karlw\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\karlw\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
C:\Users\karlw\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\karlw\AppData\Local\Programs\Python\Python311\Lib\site-p

Saved: questionnaire_data_other_factors.csv
             score     factor         type     value  association    n
0  dass_depression     gender  categorical  männlich     0.222559   40
1  dass_depression     gender  categorical  weiblich    -0.026261  339
2  dass_depression  birthyear      numeric      None    -0.060227  379
3  dass_depression     weight      numeric      None     0.116570  379
4  dass_depression   pregnant  categorical      nein     0.027391  336
